# Etapa 2 — Entrenamiento seguro con `Pipeline` y `GridSearchCV`
**Autores:** Edwin Yunis y Jairo Serrano  
**Maestría en Ingeniería Industrial · Machine Learning · Dr. Lihki Rubio**

Se comparan SVC, regresión logística, bosque aleatorio, KNN y gradient boosting. Se realiza separación 80/20 antes del aprendizaje de cualquier transformación. La selección del estimador final se hace por AUC media de validación cruzada en entrenamiento.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd()/"src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.data import load_data
from src.modeling import fit_all
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay

## Metodología y control de fuga
1. Se separa primero un 20 % de registros como conjunto de prueba estratificado (`SEED=42`).
2. Cada algoritmo incluye en `Pipeline` imputación de medianas, escalado numérico y codificación one-hot para los predictores categóricos.
3. `GridSearchCV` prueba configuraciones con cinco folds estratificados, reservando el test para evaluación final.
4. Se ordenan los candidatos por AUC de CV, **no por AUC del test**. Accuracy y matriz de confusión complementan el análisis.

In [ ]:
df = load_data(ROOT / "data" / "heart.csv")
resultados = fit_all(df)
ranking = resultados["ranking"]
display(ranking)

## Curva ROC y matriz de confusión del modelo elegido
El modelo se eligió antes de observar el rendimiento del conjunto de prueba. La curva ROC representa sensibilidad frente a la tasa de falsos positivos a distintos umbrales; la matriz de confusión muestra los errores al umbral fijo de 0,5.

In [ ]:
model = resultados["best_estimator"]
x_test, y_test = resultados["x_test"], resultados["y_test"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
RocCurveDisplay.from_estimator(model, x_test, y_test, ax=axes[0])
ConfusionMatrixDisplay.from_estimator(model, x_test, y_test, ax=axes[1], values_format="d")
axes[0].set_title("Curva ROC")
axes[1].set_title("Matriz de confusión")
plt.tight_layout()
plt.show()

## Discusión
Una AUC alta mide capacidad discriminativa dentro de este conjunto histórico, no la precisión de una predicción de insuficiencia cardíaca futura. Accuracy se interpreta teniendo en cuenta el balance de clases. La evaluación interna no reemplaza validación externa ni calibración; tampoco se permite elegir parámetros o modelos a partir del test.

### Reproducibilidad y despliegue
Ejecutar `python -m scripts.train` desde la raíz crea `artifacts/model.joblib`, el ranking real, las figuras y el informe Markdown, que alimentan `docs/index.html`. Después es posible probar `app/api.py`, Docker, Minikube, CI y Evidently conforme al README.